Importing necessary libraries:

In [24]:
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

Defining relative paths:

In [ ]:
DATA_DIR = "../data/raw/"
HEART_PATH = os.path.join(DATA_DIR, "heart_india.csv")
DIABETES_PATH = os.path.join(DATA_DIR, "diabetes_india.csv")
LIFESTYLE_PATH = os.path.join(DATA_DIR, "lifestyle_synthetic.csv")

Reading Datasets

In [ ]:
datasets = {}

datasets['heart'] = pd.read_csv(HEART_PATH)
print(f"Indian Heart Dataset: {datasets['heart'].shape[0]} rows, {datasets['heart'].shape[1]} columns")

datasets['diabetes'] = pd.read_csv(DIABETES_PATH)
print(f"Indian Diabetes Dataset: {datasets['diabetes'].shape[0]} rows, {datasets['diabetes'].shape[1]} columns")

datasets['lifestyle'] = pd.read_csv(LIFESTYLE_PATH)
print(f"Synthetic Lifestyle Dataset: {datasets['lifestyle'].shape[0]} rows, {datasets['lifestyle'].shape[1]} columns")

--- DATA INGESTION ---
Indian Heart Dataset: 10000 rows, 26 columns
Indian Diabetes Dataset: 5292 rows, 27 columns
Synthetic Lifestyle Dataset: 10000 rows, 51 columns


Checking Missing values in dataset

In [17]:
for name, df in datasets.items():
    missing_total = df.isnull().sum().sum()
    if missing_total > 0:
        print(f"{name.capitalize()} Data: {missing_total} total missing values detected.")
        print(df.isnull().sum().sort_values(ascending=False))
    else:
        print(f"{name.capitalize()} Data: Perfectly clean (0 missing values).")

Heart Data: Perfectly clean (0 missing values).
Diabetes Data: 1780 total missing values detected.
Alcohol_Intake                       1780
Age                                     0
Heart_Rate                              0
Thyroid_Condition                       0
C_Protein_Level                         0
Vitamin_D_Level                         0
Glucose_Tolerance_Test_Result           0
Polycystic_Ovary_Syndrome               0
Pregnancies                             0
Medication_For_Chronic_Conditions       0
Regular_Checkups                        0
Health_Insurance                        0
Urban_Rural                             0
Waist_Hip_Ratio                         0
HBA1C                                   0
Gender                                  0
Postprandial_Blood_Sugar                0
Fasting_Blood_Sugar                     0
Cholesterol_Level                       0
Hypertension                            0
Stress_Level                            0
Smoking_Status     

Inspecting Alcohol_Intake Column in Diabetes dataset for missing values

In [18]:
#diabetes dataset missing values
df_diabetes = pd.read_csv(DIABETES_PATH)
df_diabetes['Alcohol_Intake'].value_counts()

Alcohol_Intake
High        1764
Moderate    1748
Name: count, dtype: int64

Alcohol_Intake column has only 2 unique values "High" and "Moderate". All other similar features had three categories of "Low", "Medium", "High". Found that the missing values (NaN) was actually representing the values for "Low" attribute.

Hypothesis testing for confirming my assumpsion - misrepresentation of 'Low':

In [19]:
# Step 1: Verify the "Rule of 3" across other categorical features
print("1. Checking unique categories across other features:\n")
categorical_cols = df_diabetes.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_diabetes[col].nunique() <= 5:
        print(f"- {col}: {df_diabetes[col].unique()}")

# Step 2: Check the exact breakdown of Alcohol_Intake
print("\n2. Alcohol_Intake Value Counts (Including Missing):")
print(df_diabetes['Alcohol_Intake'].value_counts(dropna=False))

1. Checking unique categories across other features:

- Gender: ['Male' 'Other' 'Female']
- Family_History: ['No' 'Yes']
- Physical_Activity: ['High' 'Medium' 'Low']
- Diet_Type: ['Non-Vegetarian' 'Vegetarian' 'Vegan']
- Smoking_Status: ['Never' 'Current' 'Former']
- Alcohol_Intake: [nan 'Moderate' 'High']
- Stress_Level: ['Medium' 'High' 'Low']
- Hypertension: ['Yes' 'No']
- Urban_Rural: ['Urban' 'Rural']
- Health_Insurance: ['No' 'Yes']
- Regular_Checkups: ['No' 'Yes']
- Medication_For_Chronic_Conditions: ['No' 'Yes']
- Polycystic_Ovary_Syndrome: ['0' 'No' 'Yes']
- Thyroid_Condition: ['Yes' 'No']
- Diabetes_Status: ['Yes' 'No']

2. Alcohol_Intake Value Counts (Including Missing):
Alcohol_Intake
NaN         1780
High        1764
Moderate    1748
Name: count, dtype: int64


If Nan actually means "Low/None", the diabetes rate for the Nan group should be mathematically distinct from the "High" group.

Biological Correlation:

In [21]:
target_col = 'Diabetes_Status'

print(f"\n3. {target_col} Distribution by Alcohol Intake:")
# We group by the alcohol intake, filling NaNs with a temporary label to see them
rates = df_diabetes.groupby(df_diabetes['Alcohol_Intake'].fillna('MISSING (Null)'))[target_col].value_counts(normalize=True).unstack()
    
rates = (rates * 100).round(1)
print(rates)


3. Diabetes_Status Distribution by Alcohol Intake:
Diabetes_Status    No   Yes
Alcohol_Intake             
High             49.5  50.5
MISSING (Null)   49.2  50.8
Moderate         48.0  52.0


As you can see above, my assumptions were true.

As you can see below, I also found out in the "Polycystic_Ovary_Syndrome" column of the same diabetes dataset, there are values "0" and "No" which meant the same.

In [22]:
df_diabetes['Polycystic_Ovary_Syndrome'].value_counts()

Polycystic_Ovary_Syndrome
0      3534
No      909
Yes     849
Name: count, dtype: int64

Cleaning "Alcohol_Intake" and "Polycystic_Ovary_Syndrome" Columns:

In [29]:

df_diabetes = datasets['diabetes'].copy()

# Fix 1: The Missing 'Low' Alcohol Category
df_diabetes['Alcohol_Intake'] = df_diabetes['Alcohol_Intake'].fillna('Low')
print("Alcohol_Intake NaNs replaced with 'Low'.")

# Fix 2: The PCOS Data Entry Error
df_diabetes['Polycystic_Ovary_Syndrome'] = df_diabetes['Polycystic_Ovary_Syndrome'].replace('0', 'No')
print("PCOS data entry error ('0' -> 'No') fixed.")

Alcohol_Intake NaNs replaced with 'Low'.
PCOS data entry error ('0' -> 'No') fixed.


Encode Categorical Variables for the Diabetes Dataset

In [30]:
categorical_cols_diabetes = df_diabetes.select_dtypes(include=['object']).columns
le_dict_diabetes = {}

for col in categorical_cols_diabetes:
    le = LabelEncoder()
    df_diabetes[col] = le.fit_transform(df_diabetes[col].astype(str))
    le_dict_diabetes[col] = le

print(f"Diabetes Data Cleaned & Encoded. Shape: {df_diabetes.shape}")

Diabetes Data Cleaned & Encoded. Shape: (5292, 27)


Preparing Lifestyle Data for KMEANS

In [31]:
df_lifestyle = datasets['lifestyle'].copy()

# Drop identifiers
cols_to_drop = ['Country'] 
df_lifestyle = df_lifestyle.drop(columns=[c for c in cols_to_drop if c in df_lifestyle.columns])

# Encode Categorical Variables
categorical_cols_lifestyle = df_lifestyle.select_dtypes(include=['object']).columns
le_dict_lifestyle = {}

for col in categorical_cols_lifestyle:
    le = LabelEncoder()
    df_lifestyle[col] = le.fit_transform(df_lifestyle[col].astype(str))
    le_dict_lifestyle[col] = le

# Scale the data (Crucial for K-Means distance calculations)
scaler_lifestyle = StandardScaler()
df_lifestyle_scaled = pd.DataFrame(
    scaler_lifestyle.fit_transform(df_lifestyle), 
    columns=df_lifestyle.columns
)
print(f"Lifestyle Data Encoded & Scaled. Shape: {df_lifestyle_scaled.shape}")

Lifestyle Data Encoded & Scaled. Shape: (10000, 50)


Preparing Heart Dataset

In [32]:
df_heart = datasets['heart'].copy()
categorical_cols_heart = df_heart.select_dtypes(include=['object']).columns
le_dict_heart = {}

for col in categorical_cols_heart:
    le = LabelEncoder()
    df_heart[col] = le.fit_transform(df_heart[col].astype(str))
    le_dict_heart[col] = le

print(f"Heart Data Cleaned & Encoded. Shape: {df_heart.shape}")

Heart Data Cleaned & Encoded. Shape: (10000, 26)


Update our master dictionary

In [33]:
datasets['diabetes_clean'] = df_diabetes
datasets['lifestyle_scaled'] = df_lifestyle_scaled
datasets['heart_clean'] = df_heart

print("\n ALL DATASETS READY FOR MACHINE LEARNING INFERENCE.")


 ALL DATASETS READY FOR MACHINE LEARNING INFERENCE.


Heart & Lifestyle dataset categorical values inspection

In [34]:
print("\n1. HEART DATASET CATEGORIES:")
cat_heart = datasets['heart_clean'].select_dtypes(include=['object']).columns
for col in cat_heart:
    if datasets['heart_clean'][col].nunique() <= 5:
        print(f"- {col}: {datasets['heart_clean'][col].unique()}")

print("\n2. LIFESTYLE DATASET CATEGORIES (Sample):")
cat_life = datasets['lifestyle_scaled'].select_dtypes(include=['object']).columns
for col in cat_life[:5]: 
    if datasets['lifestyle_scaled'][col].nunique() <= 5:
        print(f"- {col}: {datasets['lifestyle_scaled'][col].unique()}")


1. HEART DATASET CATEGORIES:

2. LIFESTYLE DATASET CATEGORIES (Sample):
